# Spot-Check Modal Evaluation Runs

**Environment:** Run this notebook in the `cs224r-hw2` conda environment.

This notebook downloads output files from a Modal Volume and lets you inspect questions, responses, and scores for any evaluation run (AIME, MATH, or GSM8K).

## How to use
1. Set `DATASET`, `MODEL`, and `OUTPUT_TAG` in **Cell 2** to match your run.
2. Run all cells in order (`Run All`).
3. Scroll to the bottom to inspect individual problems.

In [2]:
import json
import os
import subprocess
from pathlib import Path

import pandas as pd
import numpy as np

# Verify Modal CLI is available
result = subprocess.run(["which", "modal"], capture_output=True, text=True)
if result.returncode != 0:
    raise RuntimeError("Modal CLI not found. Install with: pip install modal")

print("Modal CLI found:", result.stdout.strip())

Modal CLI found: /home/chung/miniconda3/envs/cs224r-hw2/bin/modal


## 1. Configuration — Edit these variables to match your run

In [3]:
# Which dataset / run do you want to inspect?
DATASET = "aime"          # 'aime' | 'math' | 'gsm8k'
MODEL = "qwen"            # 'qwen' | 'e3'
OUTPUT_TAG = "n2_l32768"  # e.g. 'n2_l32768', 'smoke', 'n16_l32768', etc.

# Volume settings
VOLUME_NAME = "e3-generation-vol"
VOLUME_BASE_DIR = "/data/aime_eval"  # both modal_eval_aime.py and modal_eval_general.py write here

# Where to download files locally
LOCAL_DOWNLOAD_DIR = Path("./modal_spotcheck_downloads")

# For AIME runs from modal_eval_aime.py, the parquet prefix is 'aime2025' instead of 'aime'.
# Set to True if your run used modal_eval_aime.py (not modal_eval_general.py).
IS_LEGACY_AIME = (DATASET == "aime" and not OUTPUT_TAG.startswith("n"))
# ^ heuristic: legacy runs often use custom tags like 'smoke'; general.py uses 'n{k}_l{len}' by default.
# If unsure, you can override the parquet prefix manually below.

PARQUET_PREFIX = DATASET  # general.py uses dataset name as prefix
if DATASET == "aime" and IS_LEGACY_AIME:
    PARQUET_PREFIX = "aime2025"

print(f"Dataset: {DATASET}")
print(f"Model: {MODEL}")
print(f"Tag: {OUTPUT_TAG}")
print(f"Parquet prefix: {PARQUET_PREFIX}")

Dataset: aime
Model: qwen
Tag: n2_l32768
Parquet prefix: aime


## 2. Download files from Modal Volume

In [4]:
LOCAL_DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)

# Build remote file names
# Note: modal_eval_general.py includes dataset_key in metrics/per_problem filenames;
#       modal_eval_aime.py does NOT include dataset_key.
if IS_LEGACY_AIME:
    metrics_name = f"metrics_{MODEL}_{OUTPUT_TAG}.json"
    per_problem_name = f"per_problem_{MODEL}_{OUTPUT_TAG}.csv"
else:
    metrics_name = f"metrics_{DATASET}_{MODEL}_{OUTPUT_TAG}.json"
    per_problem_name = f"per_problem_{DATASET}_{MODEL}_{OUTPUT_TAG}.csv"

parquet_name = f"{PARQUET_PREFIX}_{MODEL}_{OUTPUT_TAG}_outputs.parquet"

remote_files = [
    f"{VOLUME_BASE_DIR}/{metrics_name}",
    f"{VOLUME_BASE_DIR}/{per_problem_name}",
    f"{VOLUME_BASE_DIR}/{parquet_name}",
]

local_paths = {}
for remote_path in remote_files:
    local_path = LOCAL_DOWNLOAD_DIR / os.path.basename(remote_path)
    cmd = ["modal", "volume", "get", VOLUME_NAME, remote_path, str(local_path)]
    print(f"Downloading {os.path.basename(remote_path)} ...")
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        print(f"  ERROR: {result.stderr}")
    else:
        print(f"  -> {local_path}")
        local_paths[os.path.basename(remote_path)] = local_path

if not local_paths:
    raise RuntimeError("No files were downloaded. Check your VOLUME_NAME, DATASET, MODEL, and OUTPUT_TAG.")

  ERROR: ╭─ Error ──────────────────────────────────────────────────────────────────────╮
│ No such file or directory                                                    │
╰──────────────────────────────────────────────────────────────────────────────╯

  ERROR: ╭─ Error ──────────────────────────────────────────────────────────────────────╮
│ No such file or directory                                                    │
╰──────────────────────────────────────────────────────────────────────────────╯

  ERROR: ╭─ Error ──────────────────────────────────────────────────────────────────────╮
│ No such file or directory                                                    │
╰──────────────────────────────────────────────────────────────────────────────╯



RuntimeError: No files were downloaded. Check your VOLUME_NAME, DATASET, MODEL, and OUTPUT_TAG.

## 3. Load & Summarize Metrics

In [ ]:
# Load metrics JSON
metrics_path = local_paths.get(metrics_name)
if metrics_path and metrics_path.exists():
    with open(metrics_path) as f:
        metrics = json.load(f)
    print("=== Eval Summary ===")
    for k, v in metrics.items():
        print(f"  {k}: {v}")
else:
    print(f"Metrics file not found: {metrics_name}")

In [ ]:
# Load per-problem CSV
per_problem_path = local_paths.get(per_problem_name)
if per_problem_path and per_problem_path.exists():
    per_problem_df = pd.read_csv(per_problem_path)
    display(per_problem_df)
else:
    print(f"Per-problem file not found: {per_problem_name}")

## 4. Load Output Parquet & Inspect Individual Problems

In [ ]:
# Load the full outputs
parquet_path = local_paths.get(parquet_name)
if parquet_path and parquet_path.exists():
    outputs_df = pd.read_parquet(parquet_path)
    print(f"Loaded {len(outputs_df)} problems from {parquet_name}")
    print(f"Columns: {outputs_df.columns.tolist()}")
else:
    raise FileNotFoundError(f"Parquet not found: {parquet_name}")

### Pick a problem to inspect

In [ ]:
PROBLEM_IDX = 0  # <-- change this to inspect a different problem

row = outputs_df.iloc[PROBLEM_IDX]

# Extract question text
prompt = row["prompt"]
if isinstance(prompt, list) and len(prompt) > 0 and isinstance(prompt[0], dict):
    question_text = prompt[0].get("content", str(prompt))
else:
    question_text = str(prompt)

# Extract ground truth
reward_model = row.get("reward_model", {})
if isinstance(reward_model, dict):
    ground_truth = reward_model.get("ground_truth", "N/A")
else:
    ground_truth = "N/A"

# Extract responses (may be ndarray)
responses = row["responses"]
if hasattr(responses, "tolist"):
    responses = responses.tolist()
if not isinstance(responses, list):
    responses = [responses]

print(f"=== PROBLEM {PROBLEM_IDX} ===")
print("\n--- QUESTION ---")
print(question_text)
print("\n--- GROUND TRUTH ---")
print(ground_truth)
print(f"\n--- NUMBER OF SAMPLES: {len(responses)} ---")
for i, resp in enumerate(responses):
    print(f"\n--- RESPONSE {i + 1} (last 300 chars) ---")
    text = str(resp)
    print(text[-300:] if len(text) > 300 else text)
    print("-" * 40)

### Side-by-side comparison of all samples for one problem

In [ ]:
# Set this to the problem you want to compare across samples
COMPARE_PROBLEM_IDX = 0
SHOW_FULL_RESPONSE = False  # Set True to show full text (can be very long)
TRUNCATE_LEN = 500

row = outputs_df.iloc[COMPARE_PROBLEM_IDX]
responses = row["responses"]
if hasattr(responses, "tolist"):
    responses = responses.tolist()
if not isinstance(responses, list):
    responses = [responses]

comparison_rows = []
for i, resp in enumerate(responses):
    text = str(resp)
    display_text = text if SHOW_FULL_RESPONSE else (text[-TRUNCATE_LEN:] if len(text) > TRUNCATE_LEN else text)
    comparison_rows.append({
        "Sample": i + 1,
        "Length": len(text),
        "Last chars": display_text,
    })

comparison_df = pd.DataFrame(comparison_rows)
display(comparison_df)

# Also show the prompt for reference
prompt = row["prompt"]
if isinstance(prompt, list) and len(prompt) > 0:
    print("\n=== PROMPT ===")
    print(prompt[0].get("content", str(prompt)))

## 5. Optional: Re-score a single response locally (for debugging)

If you want to test how the scorer behaves on a specific response string, run the cell below.

In [ ]:
# Uncomment and edit the response you want to test:

# test_response = responses[0]  # use the first response from the problem above
# test_ground_truth = ground_truth

# from verl.utils.reward_score.curriculum_math.compute_score import compute_score
# score = compute_score(test_response, test_ground_truth)
# print(f"Score: {score}")